# Model 2: LSTM & GRU for Sentiment Analysis

Recurrent Neural Networks (RNNs) are designed for **sequential data** like text.
They process one token at a time while maintaining a hidden state (memory).

- **LSTM** (Long Short-Term Memory): uses gates to control what to remember/forget → handles long-range dependencies
- **GRU** (Gated Recurrent Unit): simplified LSTM with fewer parameters → often equally effective, faster to train

Both models are trained with **pretrained Word2Vec embeddings** (via Gensim).

In [ ]:
# ─────────────────────────────────────────────
# CELL 1: Install dependencies
# ─────────────────────────────────────────────
!pip install nltk gensim tensorflow kagglehub -q

In [ ]:
# ─────────────────────────────────────────────
# CELL 2: Imports
# ─────────────────────────────────────────────
import kagglehub
import pandas as pd
import numpy as np
import re, json
import nltk
import matplotlib.pyplot as plt
import seaborn as sns

# Keras / TensorFlow
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

# Gensim for Word2Vec pretrained embeddings
import gensim.downloader as gensim_api

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, classification_report,
    confusion_matrix, roc_auc_score, roc_curve
)

nltk.download('stopwords')
from nltk.corpus import stopwords
STOPWORDS = set(stopwords.words('english'))

print("TensorFlow:", tf.__version__)

In [ ]:
# ─────────────────────────────────────────────
# CELL 3: Load & Preprocess Dataset
# ─────────────────────────────────────────────
path = kagglehub.dataset_download("lakshmi25npathi/imdb-dataset-of-50k-movie-reviews")
data = pd.read_csv(f"{path}/IMDB Dataset.csv")
data = data.drop_duplicates(subset='review').reset_index(drop=True)

def clean_text(text):
    """Tokenization-friendly cleaning — keeps sentiment-bearing words."""
    text = text.lower()
    text = re.sub(r"<.*?>", "", text)        # remove HTML
    text = re.sub(r"[^a-z\s]", "", text)     # keep only letters
    text = re.sub(r"\s+", " ", text).strip() # collapse spaces
    return text

data['clean'] = data['review'].apply(clean_text)
data['label'] = data['sentiment'].map({'positive': 1, 'negative': 0})

print(f"Dataset: {data.shape}")
data.head()

In [ ]:
# ─────────────────────────────────────────────
# CELL 4: Tokenization & Padding
#
# Keras Tokenizer converts text → integer sequences.
# pad_sequences ensures all sequences have the same length
# (shorter sequences are zero-padded at the end).
# ─────────────────────────────────────────────
VOCAB_SIZE   = 20000    # keep top 20k most frequent words
MAX_LEN      = 256      # max words per review (truncate/pad to this)
EMBEDDING_DIM = 100     # Word2Vec dimension (Google News uses 300; word2vec-google-news uses 100 here for speed)

# Build vocabulary from training data only
texts = data['clean'].tolist()
labels = data['label'].values

# Split BEFORE tokenizing so test data stays unseen
texts_temp, texts_test, y_temp, y_test = train_test_split(
    texts, labels, test_size=0.2, random_state=42, stratify=labels
)
texts_train, texts_val, y_train, y_val = train_test_split(
    texts_temp, y_temp, test_size=0.25, random_state=42, stratify=y_temp
)

# Fit tokenizer ONLY on training texts
tokenizer = Tokenizer(num_words=VOCAB_SIZE, oov_token='<OOV>')
tokenizer.fit_on_texts(texts_train)

# Convert texts → integer sequences
train_seq = tokenizer.texts_to_sequences(texts_train)
val_seq   = tokenizer.texts_to_sequences(texts_val)
test_seq  = tokenizer.texts_to_sequences(texts_test)

# Pad / truncate to MAX_LEN (post-padding, pre-truncation)
X_train = pad_sequences(train_seq, maxlen=MAX_LEN, padding='post', truncating='pre')
X_val   = pad_sequences(val_seq,   maxlen=MAX_LEN, padding='post', truncating='pre')
X_test  = pad_sequences(test_seq,  maxlen=MAX_LEN, padding='post', truncating='pre')

print(f"Train: {X_train.shape} | Val: {X_val.shape} | Test: {X_test.shape}")

In [ ]:
# ─────────────────────────────────────────────
# CELL 5: Load Pretrained Word2Vec Embeddings
#
# Gensim provides 'glove-wiki-gigaword-100' (100-dim GloVe)
# and 'word2vec-google-news-300' (300-dim Word2Vec).
# We use GloVe-100 for speed; swap for Word2Vec-300 for best results.
#
# The embedding matrix maps each word index → its pretrained vector.
# Words not in the pretrained vocab get a random small vector.
# ─────────────────────────────────────────────
print("Downloading pretrained GloVe embeddings (may take a few minutes)...")
word_vectors = gensim_api.load('glove-wiki-gigaword-100')  # 100-dim GloVe
print("Embeddings loaded!")

word_index = tokenizer.word_index  # {word: integer_index}

# Build embedding matrix: row i = vector for word at index i
embedding_matrix = np.zeros((VOCAB_SIZE + 1, EMBEDDING_DIM))
found = 0
for word, idx in word_index.items():
    if idx <= VOCAB_SIZE:               # only keep top VOCAB_SIZE words
        if word in word_vectors:        # check if word exists in GloVe
            embedding_matrix[idx] = word_vectors[word]
            found += 1
        # words NOT in GloVe stay as zeros (will be learned during fine-tuning)

print(f"Words found in GloVe: {found} / {min(len(word_index), VOCAB_SIZE)}")
print(f"Embedding matrix shape: {embedding_matrix.shape}")

In [ ]:
# ─────────────────────────────────────────────
# CELL 6: Helper — Embedding Layer Factory
#
# trainable=False freezes GloVe weights (faster, less overfit)
# trainable=True fine-tunes them on IMDB (usually slightly better)
# ─────────────────────────────────────────────
def make_embedding_layer(trainable=True):
    """Returns a Keras Embedding layer initialized with pretrained weights."""
    return layers.Embedding(
        input_dim=VOCAB_SIZE + 1,
        output_dim=EMBEDDING_DIM,
        weights=[embedding_matrix],     # initialize with GloVe
        input_length=MAX_LEN,
        trainable=trainable,             # allow fine-tuning
        name='embedding'
    )

In [ ]:
# ─────────────────────────────────────────────
# CELL 7: Build LSTM Model
#
# Architecture:
#   Embedding → Bidirectional LSTM → LSTM → Dense → Output
#
# Bidirectional LSTM reads the sequence both forward and backward,
# capturing context from both directions (useful for sentiment).
# ─────────────────────────────────────────────
def build_lstm(units=64, dropout=0.3, learning_rate=1e-3):
    """
    Bidirectional LSTM sentiment classifier.

    Args:
        units        : LSTM hidden units per direction
        dropout      : recurrent + standard dropout rate
        learning_rate: Adam learning rate
    """
    model = keras.Sequential(name='BiLSTM')

    # Pretrained embedding layer
    model.add(make_embedding_layer(trainable=True))

    # SpatialDropout1D drops entire feature maps (more effective than regular dropout for embeddings)
    model.add(layers.SpatialDropout1D(dropout))

    # Bidirectional LSTM:
    #   return_sequences=True → pass full sequence to next LSTM layer
    #   recurrent_dropout     → dropout on hidden-to-hidden connections
    model.add(layers.Bidirectional(
        layers.LSTM(units,
                    return_sequences=True,
                    dropout=dropout,
                    recurrent_dropout=0.1),
        name='bidir_lstm_1'
    ))

    # Second LSTM layer — reads the output of the first
    # return_sequences=False → only output last timestep
    model.add(layers.LSTM(units // 2,
                           dropout=dropout,
                           recurrent_dropout=0.1,
                           name='lstm_2'))

    # Dense layers for classification head
    model.add(layers.Dense(32, activation='relu'))
    model.add(layers.Dropout(dropout))
    model.add(layers.Dense(1, activation='sigmoid', name='output'))

    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=learning_rate),
        loss='binary_crossentropy',
        metrics=['accuracy']
    )
    return model

lstm_model = build_lstm()
lstm_model.summary()

In [ ]:
# ─────────────────────────────────────────────
# CELL 8: Build GRU Model
#
# GRU is similar to LSTM but combines forget/input gates
# into a single 'update gate', making it faster to train
# with comparable performance.
# ─────────────────────────────────────────────
def build_gru(units=64, dropout=0.3, learning_rate=1e-3):
    """
    Bidirectional GRU sentiment classifier.
    """
    model = keras.Sequential(name='BiGRU')

    model.add(make_embedding_layer(trainable=True))
    model.add(layers.SpatialDropout1D(dropout))

    # Bidirectional GRU
    model.add(layers.Bidirectional(
        layers.GRU(units,
                   return_sequences=True,
                   dropout=dropout,
                   recurrent_dropout=0.1),
        name='bidir_gru_1'
    ))

    # Second GRU layer
    model.add(layers.GRU(units // 2,
                          dropout=dropout,
                          recurrent_dropout=0.1,
                          name='gru_2'))

    model.add(layers.Dense(32, activation='relu'))
    model.add(layers.Dropout(dropout))
    model.add(layers.Dense(1, activation='sigmoid', name='output'))

    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=learning_rate),
        loss='binary_crossentropy',
        metrics=['accuracy']
    )
    return model

gru_model = build_gru()
gru_model.summary()

In [ ]:
# ─────────────────────────────────────────────
# CELL 9: Train Both Models
# ─────────────────────────────────────────────
callbacks = [
    EarlyStopping(monitor='val_loss', patience=4, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=2, verbose=1, min_lr=1e-6)
]

BATCH_SIZE = 128
EPOCHS     = 15

print("\n── Training LSTM ──")
lstm_history = lstm_model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=EPOCHS, batch_size=BATCH_SIZE,
    callbacks=callbacks, verbose=1
)

print("\n── Training GRU ──")
gru_history = gru_model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=EPOCHS, batch_size=BATCH_SIZE,
    callbacks=callbacks, verbose=1
)

In [ ]:
# ─────────────────────────────────────────────
# CELL 10: Training Curves Comparison
# ─────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for label, hist in [('LSTM', lstm_history), ('GRU', gru_history)]:
    axes[0].plot(hist.history['val_loss'],     label=f'{label} Val Loss')
    axes[1].plot(hist.history['val_accuracy'], label=f'{label} Val Acc')

axes[0].set_title('Validation Loss')
axes[0].set_xlabel('Epoch'); axes[0].legend()
axes[1].set_title('Validation Accuracy')
axes[1].set_xlabel('Epoch'); axes[1].legend()

plt.tight_layout()
plt.savefig('rnn_training_curves.png', dpi=150)
plt.show()

In [ ]:
# ─────────────────────────────────────────────
# CELL 11: Evaluate Both Models on Test Set
# ─────────────────────────────────────────────
results = {}

for model_name, model in [('LSTM', lstm_model), ('GRU', gru_model)]:
    y_prob = model.predict(X_test).flatten()
    y_pred = (y_prob >= 0.5).astype(int)

    acc    = accuracy_score(y_test, y_pred)
    auc    = roc_auc_score(y_test, y_prob)
    report = classification_report(y_test, y_pred,
                                   target_names=['negative','positive'])

    print(f"\n{'='*40}")
    print(f"  {model_name} Results")
    print(f"{'='*40}")
    print(f"  Test Accuracy : {acc:.4f}")
    print(f"  ROC-AUC       : {auc:.4f}")
    print(report)

    results[model_name] = {
        'test_accuracy': round(float(acc), 4),
        'roc_auc': round(float(auc), 4),
        'classification_report': report
    }

    # Confusion matrix
    cm = confusion_matrix(y_test, y_pred)
    plt.figure(figsize=(6, 5))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=['Negative','Positive'],
                yticklabels=['Negative','Positive'])
    plt.title(f'Confusion Matrix – {model_name}')
    plt.xlabel('Predicted'); plt.ylabel('Actual')
    plt.tight_layout()
    plt.savefig(f'{model_name.lower()}_confusion_matrix.png', dpi=150)
    plt.show()

# Save results
results['hyperparameters'] = {
    'vocab_size': VOCAB_SIZE, 'max_len': MAX_LEN,
    'embedding_dim': EMBEDDING_DIM, 'pretrained_embeddings': 'GloVe-100',
    'lstm_units': 64, 'gru_units': 64,
    'dropout': 0.3, 'batch_size': BATCH_SIZE, 'optimizer': 'Adam'
}

with open('results_model2_rnn.json', 'w') as f:
    json.dump(results, f, indent=2)

print("\nResults saved to results_model2_rnn.json")

lstm_model.save('model2_lstm.keras')
gru_model.save('model2_gru.keras')
print("Models saved!")